# 4. Analysis

**Step 4 of 4**: codebook -> download & parse -> accelerometry (optional) -> analysis.

Five things worth doing with the parsed table, on one example: the complete blood count (CBC)
of **middle-aged participants (20-60) of the 2009-2010 survey**, against age, gender and
mortality.

1. **Slicing** - taking the topics, the survey and the age band you want
2. **Availability** - which survey measured what
3. **Correlation** - what moves together, and what a p-value says about it
4. **Clustering** - grouping the CBC variables, not the people
5. **Survival** - a Cox model of all-cause mortality, trained and tested

Needs `./CSV/nhanes_userdata.csv.gz` from `parse_userdata.ipynb`, and
`pip install lifelines scikit-learn`.

## 1. Slicing the table

The parsed table has two header rows, so the **topic** is a slice of its own: asking for
`["Demographic", "Blood Count", "Mortality"]` keeps three blocks of columns and drops the
other twenty-two. The survey is an ordinary column, so one cycle is an ordinary filter.

Working within one survey is worth doing on purpose. Every survey asked a slightly different
set of questions, and stacking twelve of them gives a sample size so large that every
correlation becomes "significant".

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pynhanes

userdata = pd.read_csv("./CSV/nhanes_userdata.csv.gz", sep=";", index_col=0, header=[0, 1],
                       low_memory=False)
cbc = list(userdata["Blood Count"].columns)

# take three topics, then one survey
data = userdata[["Demographic", "Blood Count", "Mortality"]]
data = data[data[("Demographic", "Survey")] == "2009-2010"]
data.columns = [name for _, name in data.columns]      # one header row is easier from here

print(f"{len(data)} participants of 2009-2010, {data.shape[1]} columns, "
      f"{len(cbc)} of them blood count")

## 2. What each survey measured

Before settling on one cycle, it is worth seeing where the blood count exists at all.

In [ ]:
available = (userdata.groupby(("Demographic", "Survey"))["Blood Count"]
             .apply(lambda g: 100 * g.notna().mean()).T)

plt.figure(figsize=(8, 6), facecolor="white")
plt.imshow(available.values, aspect="auto", cmap="viridis", vmin=0, vmax=100)
plt.colorbar(label="participants with a value, %")
plt.xticks(range(available.shape[1]), available.columns, rotation=90)
plt.yticks(range(available.shape[0]), available.index)
plt.title("Blood count availability per survey")
plt.tight_layout()
plt.show()

Around 80% in every cycle - the blood count is drawn at the examination, which not everyone
attends. The last cycle is thinner, and the mortality linkage does not cover it at all, which
is why this notebook works on an older one.

**The availability heatmap above is the only part that looks at everybody.** From here on the
notebook keeps participants aged **20 to 60**. Narrowing an age band is the commonest way to
stop age from explaining everything: in a sample that runs from 18 to 80, age alone predicts
death so well that nothing else gets a look in. The price is paid in events - see the
survival section.

In [ ]:
AGE_MIN, AGE_MAX = 20, 60     # a middle-aged cohort

DESIGN = ["Survey PSU", "Survey strata"]
cohort = data[cbc + ["Age", "Gender", "Mortality event", "Mortality tte (yr)"] + DESIGN].dropna()
cohort = cohort[cohort["Age"].between(AGE_MIN, AGE_MAX)]

# a county is only identified by its PSU *within* a stratum, so combine the two
cohort["county"] = cohort["Survey strata"] * 10 + cohort["Survey PSU"]

print(f"cohort: {len(cohort)} participants aged {AGE_MIN}-{AGE_MAX}, "
      f"{int(cohort['Mortality event'].sum())} of them died during follow-up")
print(f"they come from {cohort['county'].nunique()} counties "
      f"in {cohort['Survey strata'].nunique()} strata")


## 3. Correlation

Blood count variables are not independent: percentages of the same white cells add up to 100,
and haemoglobin is most of what a haematocrit measures. Here the **size** of the correlation
matters, not its sign, so the plot uses the absolute value and a sequential colormap.

In [ ]:
rho = cohort[cbc].corr(method="spearman").abs()

plt.figure(figsize=(8, 7), facecolor="white")
plt.imshow(rho.values, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="|Spearman rho|")
plt.xticks(range(len(cbc)), cbc, rotation=90)
plt.yticks(range(len(cbc)), cbc)
plt.title("Blood count correlations, as the table orders them")
plt.tight_layout()
plt.show()

A **p-value** is the chance of seeing a correlation this strong if there were really no
relation at all - so a *small* p-value is the encouraging one, and below 0.05 is the usual
convention. It depends on the sample size as much as on the strength: with enough people even
a trivial correlation clears any threshold, which is exactly why this notebook stays inside
one survey.

**And the usual formula is wrong here.** NHANES cannot scatter its examination centre across
the country, so it visits about 30 counties and measures several hundred people in each. The
cohort above comes from 31 of them. `scipy.stats.spearmanr` assumes 3,968 independent people;
it really has something closer to 31 independent places, so it reports far more certainty
than the data support. `Survey PSU` and `Survey strata` are the columns that record this, and
the function below uses them - see
[Sample weights](https://pynhanes.readthedocs.io/en/latest/weights.html).


In [ ]:
def cluster_bootstrap_p(x, y, county, draws=600, seed=0):
    """p-value from resampling whole counties, not single people.

    NHANES measures a few hundred people in each of about 30 counties, and
    neighbours resemble each other. Two participants from one county are
    therefore not two independent observations, and a test that assumes they are
    reports far more certainty than the data support.

    Drawing counties with replacement keeps that structure. The spread of the
    correlation across those draws is its real uncertainty; the p-value is how
    often the draws land on the other side of zero.
    """
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(county.values == c) for c in county.unique()]
    r = np.empty(draws)
    for k in range(draws):
        take = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        r[k] = stats.spearmanr(x.values[take], y.values[take]).statistic
    return 2 * min((r <= 0).mean(), (r >= 0).mean())


pairs = []
for i in range(len(cbc)):
    for j in range(i + 1, len(cbc)):
        r, p_naive = stats.spearmanr(cohort[cbc[i]], cohort[cbc[j]])
        pairs.append((cbc[i], cbc[j], abs(r), p_naive))
pairs = pd.DataFrame(pairs, columns=["a", "b", "|rho|", "p"]).sort_values("|rho|", ascending=False)

print(f"{len(pairs)} pairs, {(pairs.p < 0.05).sum()} of them with p < 0.05\n")
print("strongest:"); print(pairs.head(3).to_string(index=False))
print("\nweakest:");  print(pairs.tail(3).to_string(index=False))

# the weakest pair the naive test still calls significant - the one where
# ignoring the counties is most likely to have invented the finding
marginal = pairs[pairs.p < 0.05].iloc[-1]
p_cluster = cluster_bootstrap_p(cohort[marginal.a], cohort[marginal.b], cohort["county"])
print(f"\nweakest 'significant' pair: {marginal.a} vs {marginal.b}  (|rho| = {marginal['|rho|']:.3f})")
print(f"   p assuming independent people : {marginal.p:.3g}")
print(f"   p resampling whole counties   : {p_cluster:.3g}")


## 4. Clustering the variables

The heatmap above is hard to read because the columns are in the order the table happens to
store them. Hierarchical clustering puts variables that move together next to each other, and
the matrix turns into blocks.

Note what is being clustered: the **20 variables**, not the 6000 people. The distance between
two variables is `1 - |rho|`, so two measurements that track each other are close.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

distance = 1 - rho.values
np.fill_diagonal(distance, 0)
tree = linkage(squareform(distance, checks=False), method="average")
order = dendrogram(tree, no_plot=True)["leaves"]
ordered = rho.iloc[order, order]

fig = plt.figure(figsize=(11, 7), facecolor="white")
grid = fig.add_gridspec(1, 2, width_ratios=[4, 1], wspace=0.05)

# origin="lower" so that row 0 sits at the bottom, where the tree puts leaf 0
ax = fig.add_subplot(grid[0])
image = ax.imshow(ordered.values, cmap="viridis", vmin=0, vmax=1, origin="lower")
ax.set_xticks(range(len(cbc)), ordered.columns, rotation=90)
ax.set_yticks(range(len(cbc)), ordered.index)
ax.set_ylim(-0.5, len(cbc) - 0.5)
ax.set_title("Same matrix, variables reordered by the tree")

# the tree leaves sit at y = 5, 15, 25 ... so the two axes line up once both
# ranges are set by hand - sharing the axis would squash the heatmap instead
tree_ax = fig.add_subplot(grid[1])
dendrogram(tree, orientation="right", ax=tree_ax, no_labels=True,
           link_color_func=lambda k: "grey")
tree_ax.set_ylim(0, 10 * len(cbc))
tree_ax.set_xticks([])
tree_ax.set_yticks([])
for side in tree_ax.spines.values():
    side.set_visible(False)
fig.colorbar(image, ax=tree_ax, label="|Spearman rho|", fraction=0.1, pad=0.2)
plt.show()

Two blocks, and they are the two halves of a blood count: the **white cells** (their
counts and their percentages of each other) and the **red cells** (their number, their size,
and how much haemoglobin they carry).

Take one representative from each - the member closest to the middle of its own group, that
is the one most correlated with the rest of it.

In [ ]:
groups = fcluster(tree, t=4, criterion="maxclust")
sizes = pd.Series(groups).value_counts()

representatives = []
for group in sizes.index[:2]:
    members = [cbc[i] for i in range(len(cbc)) if groups[i] == group]
    centre = max(members, key=lambda m: rho.loc[m, members].mean())
    representatives.append(centre)
    print(f"group of {len(members):2d}: {centre}")
    print(f"           {', '.join(members)}\n")

print(f"the two representatives correlate at only "
      f"{rho.loc[representatives[0], representatives[1]]:.2f}")

## 5. A Cox model of survival

`Mortality event` is 1 when the participant died during follow-up, and `Mortality tte (yr)`
is how long they were followed - exactly what a proportional-hazards model wants.

**Why only two blood variables.** A hazard ratio answers "what happens when this variable
moves and the others stay put", which stops meaning much when the others cannot stay put:
feed all twenty in, and the model splits one effect across a group of near-duplicates and the
individual coefficients become unreadable. Two ways out - pick a few variables that are not
correlated with each other, which is what we do here, or decorrelate them first with
something like PCA, which is beyond this example.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index

covariates = representatives + ["Age"]
model_data = pd.DataFrame(StandardScaler().fit_transform(cohort[covariates]),
                          columns=covariates, index=cohort.index)
model_data["Gender"] = cohort["Gender"].values
model_data["event"] = cohort["Mortality event"].astype(int).values
model_data["time"] = cohort["Mortality tte (yr)"].values
model_data["county"] = cohort["county"].values
model_data = model_data[model_data["time"] > 0]

# split by county, not by person: people from one county are alike, so letting a
# county appear in both halves lets the model memorise it and flatters the test score
counties = model_data["county"].unique()
rng = np.random.default_rng(0)
held_out = set(rng.choice(counties, size=max(1, len(counties) // 3), replace=False))
test = model_data[model_data["county"].isin(held_out)]
train = model_data[~model_data["county"].isin(held_out)]

# cluster_col tells lifelines the rows are grouped, so the standard errors and
# p-values count counties rather than people
cox = CoxPHFitter().fit(train, duration_col="time", event_col="event",
                        cluster_col="county")

for label, part in [("train", train), ("test", test)]:
    risk = cox.predict_partial_hazard(part.drop(columns="county"))
    c = concordance_index(part["time"], -risk, part["event"])
    print(f"concordance {label}: {c:.3f}  ({len(part)} people, {int(part['event'].sum())} deaths)")
print()
print(cox.summary[["coef", "exp(coef)", "p"]].to_string(float_format=lambda v: f"{v:.3g}"))


Concordance is the chance that, of two people, the one the model calls riskier dies first.
Here the split is **by county**, so no county appears in both halves - a person-level split
lets the model learn a county from its training half and meet it again in the test half, which
flatters the score.

`cluster_col="county"` does the same job for the p-values as the bootstrap did for the
correlations: the standard errors count counties, not people.

Every covariate is standardised, so a hazard ratio is "per standard deviation" - except
`Gender`, which is male against female.

**No weights here, and that is deliberate.** Weights answer "how common is this in the United
States". This model asks a different question - whether neutrophils carry risk *within* a
person - and for that a weighted fit costs precision without removing bias, as long as the
model already holds the variables NHANES over-sampled on. Age and gender are both in it.

**This is where the age band shows its two faces.** Across all adults the same model reaches
a concordance of about 0.86 and age carries a hazard ratio near 5 per standard deviation.
Restricted to 20-60 it reaches about 0.75 and age drops to under 3 - worse at predicting, but
the point is what happens to the blood: the neutrophil hazard ratio roughly doubles in
relative terms and becomes the clearest non-demographic signal in the model. Take out the
variable that explains the most, and the rest become visible.


In [ ]:
risk = cox.predict_partial_hazard(test)
tertile = pd.qcut(risk, 3, labels=["low", "middle", "high"])

plt.figure(figsize=(8, 5), facecolor="white")
colours = plt.get_cmap("viridis")(np.linspace(0.15, 0.85, 3))
for colour, group in zip(colours, ["low", "middle", "high"]):
    part = test[tertile == group]
    km = KaplanMeierFitter().fit(part["time"], part["event"], label=f"{group} risk")
    km.plot_survival_function(ax=plt.gca(), color=colour, ci_alpha=0.15)
    print(f"{group:7s} n={len(part):5d}  survival at 10 years: {float(km.predict(10)):.3f}")
plt.xlabel("years of follow-up")
plt.ylabel("surviving")
plt.title("Held-out test set, split by predicted risk")
plt.tight_layout()
plt.show()

Ten years on the high-risk third has lost several times as many people as the low-risk third.
Read the gap rather than the exact numbers: restricting to 20-60 left only about 160 deaths
in the whole cohort and some 50 in the test set, so each curve rests on a handful of events
and the low and middle thirds are not really distinguishable. Widening the band, pooling
several surveys, or dropping the split and fitting on everyone are the three ways to get the
power back.

One thing this example leaves out on purpose: NHANES is a **weighted** survey, and
`Sample weight` in the Demographic block is what makes an estimate represent the US
population rather than the people who happened to be examined. Nothing above uses it, so read
the numbers as a demonstration of method, not as national statistics.